# Confidence-Score Exploration

Run all 5 benchmarked models on a set of images and collect their top-1 prediction + confidence into a single wide-format DataFrame.

One row = one image. Columns:

| Column | Content |
|---|---|
| `filename` | image filename |
| `true_class` | parent directory name (ImageFolder convention) |
| `class_<model>` | top-1 predicted class for that model |
| `confidence_<model>` | associated softmax probability |

5 models × 2 cols = 10 prediction columns (+ 2 metadata columns).

Use cases enabled by this shape:
- Mean / median / std of confidence per class per model
- Inter-model agreement rate (fraction of images where all 5 models agree)
- Per-image majority voting
- Disagreement hotspots (which classes cause the models to split)
- Calibration analysis (confidence vs correctness)

**Runs both locally and on Colab** — Cell 1 auto-detects the environment.

## 1. Environment setup (Colab vs local)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# ---------- Detect environment ----------
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print('Environment:', 'Colab' if IN_COLAB else 'local')

# ---------- Colab-only setup ----------
if IN_COLAB:
    # Install runtime deps that are not preinstalled on Colab
    subprocess.run(['pip', 'install', '-q', 'timm', 'wandb', 'loguru'], check=True)

    # Mount Drive (used below for dataset + output CSV)
    from google.colab import drive
    if not Path('/content/drive').is_dir() or not any(Path('/content/drive').iterdir()):
        drive.mount('/content/drive')

    # Clone the repo so herbs_detection is importable.
    # IMPORTANT: push your latest backend refactor to GitHub before running this cell.
    REPO_ROOT = Path('/content/plant-detect')
    if not REPO_ROOT.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/jimmyouellet/plant-detect.git',
             str(REPO_ROOT)],
            check=True,
        )

    # wandb authentication — store your key once via Colab secrets (left panel → key icon)
    # named WANDB_API_KEY, then it is loaded here automatically.
    try:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except Exception as exc:
        print('Could not read WANDB_API_KEY from Colab secrets:', exc)
        print('Falling back to interactive wandb.login() — paste your key when prompted.')
        import wandb
        wandb.login()
    os.environ.setdefault('WANDB_PROJECT', 'certification')
    os.environ.setdefault('WANDB_ENTITY',  'jimmyouellet')  # change if different

# ---------- Local setup ----------
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Make the herbs_detection package importable in both environments
sys.path.insert(0, str(REPO_ROOT / 'backend' / 'app' / 'src'))

print('REPO_ROOT =', REPO_ROOT)

In [ ]:
import random
import threading

import pandas as pd
import torch
from tqdm.auto import tqdm

from herbs_detection.model_registry import MODEL_REGISTRY, ENABLED_KEYS
from herbs_detection.timm_predictor import TimmPredictor

print('Device        :', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Models to run :', ENABLED_KEYS)

## 2. Load all 5 models in parallel

`TimmPredictor.load()` pulls each artifact from wandb (or falls back to a local `<artifact>.pth`) and blocks predict calls on a `threading.Event` until weights are in memory. We load in parallel to shave startup time.

In [ ]:
# Where wandb artifacts get cached. On Colab we keep them on the local SSD (fast)
# rather than Drive (slow). They will be lost when the runtime recycles, but
# re-downloading from wandb is quick on a good connection.
CACHE_ROOT = Path('/content/wandb_cache') if IN_COLAB else (REPO_ROOT / 'models' / 'wandb')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

predictors: dict[str, TimmPredictor] = {}
threads = []

for cfg in MODEL_REGISTRY:
    if not cfg.enabled:
        continue
    p = TimmPredictor(cfg, cache_root=CACHE_ROOT)
    predictors[cfg.key] = p
    t = threading.Thread(target=p.load, daemon=True)
    t.start()
    threads.append(t)

for t in threads:
    t.join()

for key, p in predictors.items():
    if p._load_error is not None:
        raise RuntimeError(f'{key} failed to load') from p._load_error

print('All models ready:', list(predictors))

## 3. Collect image paths

Point `DATA_ROOT` at an `ImageFolder`-style directory (one subdirectory per class). Set `SAMPLE_SIZE` to an integer for a random subset, or `None` to use everything.

On Colab the dataset typically lives on Google Drive or in a tarball you have extracted to `/content`. Extract before running this cell if needed:

```python
# example: one-time extraction from a Drive tarball to local SSD
# !tar -xzf /content/drive/MyDrive/plant_detect/dataset.tar.gz -C /content/
```

In [ ]:
if IN_COLAB:
    DATA_ROOT = Path('/content/dataset/val')               # adjust
    OUTPUT_DIR = Path('/content/drive/MyDrive/plant_detect/figures')
else:
    DATA_ROOT = REPO_ROOT / 'data' / 'raw' / 'all_images'  # adjust
    OUTPUT_DIR = REPO_ROOT / 'figures'

SAMPLE_SIZE: int | None = 500  # None = use every image
RANDOM_SEED = 42
VALID_EXT   = {'.jpg', '.jpeg', '.png'}

all_paths = [
    p for p in DATA_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in VALID_EXT
]

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(all_paths):
    random.seed(RANDOM_SEED)
    all_paths = random.sample(all_paths, SAMPLE_SIZE)

all_paths.sort()  # deterministic row order
print(f'{len(all_paths)} images from {DATA_ROOT}')

## 4. Run batch prediction for each model

`predict_set` already batches internally. On a Colab T4/L4 GPU, batch_size 128–256 works comfortably for the smaller backbones; reduce if you see out-of-memory errors on EfficientNet-B4 (380×380 inputs).

In [ ]:
BATCH_SIZE = 128 if torch.cuda.is_available() else 32

path_strs = [str(p) for p in all_paths]
results_per_model: dict[str, list[tuple[str, float]]] = {}

for key, p in tqdm(predictors.items(), desc='models'):
    results_per_model[key] = p.predict_set(path_strs, batch_size=BATCH_SIZE)

print('Done.')

## 5. Assemble the DataFrame

In [ ]:
rows = {
    'filename'  : [p.name for p in all_paths],
    'true_class': [p.parent.name for p in all_paths],
}
for key, preds in results_per_model.items():
    rows[f'class_{key}']      = [cls for cls, _ in preds]
    rows[f'confidence_{key}'] = [conf for _, conf in preds]

df = pd.DataFrame(rows)
df.head()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'confidence_exploration.csv'
df.to_csv(OUTPUT_CSV, index=False)
print('Saved', OUTPUT_CSV)

## 6. Example analyses

Starters — run, adapt, replace.

### 6.1 Mean confidence per class per model

In [ ]:
conf_cols = [f'confidence_{k}' for k in predictors]
(df.groupby('true_class')[conf_cols]
   .mean()
   .round(3)
   .sort_values(conf_cols[0]))

### 6.2 Correct vs incorrect confidence (calibration signal)

In [ ]:
summary = []
for key in predictors:
    correct = df[f'class_{key}'] == df['true_class']
    summary.append({
        'model'        : key,
        'accuracy'     : correct.mean(),
        'conf_correct' : df.loc[correct, f'confidence_{key}'].mean(),
        'conf_wrong'   : df.loc[~correct, f'confidence_{key}'].mean(),
    })
pd.DataFrame(summary).round(3)

### 6.3 Inter-model agreement (row level)

In [ ]:
class_cols = [f'class_{k}' for k in predictors]
df['n_distinct_predictions'] = df[class_cols].nunique(axis=1)

agreement = df['n_distinct_predictions'].value_counts().sort_index()
agreement_pct = (agreement / len(df) * 100).round(1)
pd.DataFrame({'n_images': agreement, 'pct': agreement_pct})

### 6.4 Pairwise agreement matrix

In [ ]:
import numpy as np

keys = list(predictors)
agree = np.zeros((len(keys), len(keys)))
for i, a in enumerate(keys):
    for j, b in enumerate(keys):
        agree[i, j] = (df[f'class_{a}'] == df[f'class_{b}']).mean()
pd.DataFrame(agree, index=keys, columns=keys).round(3)

### 6.5 Hardest cases (≥3 distinct model predictions)

In [ ]:
hard = df[df['n_distinct_predictions'] >= 3].copy()
hard['mean_confidence'] = hard[conf_cols].mean(axis=1).round(3)
hard.sort_values('mean_confidence').head(20)